## 01 Test the access to the RSP APIwith pyvo 

- author : Sylvie Dagoret-Campagne
- creation date : 2026-02-28

In [1]:
import os
import pyvo
import requests

In [2]:
# 1. Token d'authentification (stocké dans une variable d'environnement)
# export RSP_TOKEN="votre_token" dans ton .zshrc ou .bashrc
token = os.environ["RSP_TOKEN"]
#print(f"token {token}")

In [3]:
# Méthode recommandée avec pyvo récent
credential = pyvo.auth.authsession.AuthSession()
credential.credentials.set("lsst-token", token)

# 2. Session HTTP authentifiée
session = requests.Session()
session.headers["Authorization"] = f"Bearer {token}"

#service = pyvo.dal.TAPService("https://data.lsst.cloud/api/tap", session=session)

In [4]:
# 3. Connexion au service TAP de Rubin
rsp_tap_url = "https://data.lsst.cloud/api/tap"
service = pyvo.dal.TAPService(rsp_tap_url, session=session)

In [5]:
# Liste uniquement les noms (plus léger, évite le bug)
table_names = list(service.tables.keys())
for name in table_names:
    print(name)

dp02_dc2_catalogs.Object
dp02_dc2_catalogs.Source
dp02_dc2_catalogs.ForcedSource
dp02_dc2_catalogs.DiaObject
dp02_dc2_catalogs.DiaSource
dp02_dc2_catalogs.ForcedSourceOnDiaObject
dp02_dc2_catalogs.ObsCore
dp02_dc2_catalogs.Visit
dp02_dc2_catalogs.CcdVisit
dp02_dc2_catalogs.CoaddPatches
dp02_dc2_catalogs.TruthSummary
dp02_dc2_catalogs.MatchesTruth
dp1.Object
dp1.Source
dp1.ForcedSource
dp1.DiaObject
dp1.DiaSource
dp1.ForcedSourceOnDiaObject
dp1.CoaddPatches
dp1.Visit
dp1.CcdVisit
dp1.MPCORB
dp1.SSObject
dp1.SSSource
ivoa.ObsCore
tap_schema.schemas
tap_schema.tables
tap_schema.columns
tap_schema.keys
tap_schema.key_columns


In [6]:
# 4. Requête ADQL (exemple : visites en bandes g et r autour de ECDFS)
query = """
SELECT visit, ra, dec, band, expMidptMJD
FROM dp1.Visit
WHERE CONTAINS(
      POINT('ICRS', ra, dec),
      CIRCLE('ICRS', 53.13, -28.10, 3)
    ) = 1
  AND band IN ('g', 'r')
ORDER BY expMidptMJD ASC
"""

In [7]:
# 5. Soumission et exécution asynchrone du job
job = service.submit_job(query)
job.run()
job.wait(phases=["COMPLETED", "ERROR"])
print("Job phase:", job.phase)
if job.phase == "ERROR":
    job.raise_if_error()

Job phase: COMPLETED


In [8]:
# 6. Récupération des résultats en table Astropy
results = job.fetch_result().to_table()
print(f"Nombre de lignes : {len(results)}")
print(results[:5])

Nombre de lignes : 467
    visit            ra               dec        band    expMidptMJD    
                    deg               deg                     d         
------------- ---------------- ----------------- ---- ------------------
2024110800246 53.3274246636802 -28.0723466054328    r  60623.25932903928
2024110800247 53.1413782855617 -28.1312112068802    r  60623.25989537033
2024110800250 53.1891579665548 -28.2085120405563    r 60623.262127748836
2024110800251 53.0158795112189 -28.0984834194087    r   60623.2626942534
2024110800254 52.9367180670944 -28.2058717329746    r   60623.2648977835
